# Week 5 — Subqueries and Window Functions: Window Functions
## Phase 2b SQL | PORA Academy Cohort 7 — **Demo**

By the end of this session, you will be able to:
- Compute rankings, running totals, and comparisons **without losing row-level detail** — the defining trait of a window function versus a `GROUP BY`
- Rank rows with `RANK() OVER (ORDER BY ...)`, and build a running total across time with `SUM(...) OVER (ORDER BY ...)`
- Explain the difference between `ROW_NUMBER()` and `RANK()` — and why that difference matters the moment two rows tie


### Run this first

The cell below loads all 8 Olist tables into a SQLite database and connects the
`%%sql` magic to it. It is the same setup cell you ran on Wednesday — run it once,
wait for `Database ready.`, and leave it alone.


In [ ]:
# =====================================================================
# Olist SQL Setup — runs on BOTH Google Colab and a local machine.
# Run this cell FIRST. It loads the 8 Olist tables into a SQLite
# database and connects the %%sql magic to it. You should not need to
# edit anything unless auto-detection fails (see the two knobs below).
#
# Design notes:
# - We teach SQL with the %%sql cell magic (jupysql), not pd.read_sql().
# - jupysql opens its OWN connection, so the DB must be a real FILE
#   (a :memory: DB would be invisible to it).
# - We use jupysql (the maintained SQL magic). On Colab we install it,
#   because Colab ships the legacy ipython-sql, which (a) can't take a
#   connection by engine variable and (b) renders every result through
#   prettytable.__dict__[style], crashing on modern prettytable with
#   KeyError 'DEFAULT'/'SINGLE_BORDER'. jupysql fixes both.
# - autopandas=True makes every %%sql result a pandas DataFrame, which
#   lets the self-check cells assert on .iloc/.shape directly.
# =====================================================================
import os, glob, sqlite3, tempfile, zipfile
import pandas as pd

# --- Optional knobs (leave blank; only set if auto-detect fails) ------
LOCAL_DATA_DIR = ""   # local run: folder that holds olist_orders_dataset.csv
DRIVE_ZIP_PATH = ""   # Colab: full path to phase-2-python-sql.zip in your Drive
# ---------------------------------------------------------------------

# Detect Colab (google.colab only imports there). Outside Colab — including
# the content-pipeline validator — this falls through to the local branch.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ON_COLAB = True
except ModuleNotFoundError:
    ON_COLAB = False


def _colab_find_zip():
    """Locate phase-2-python-sql.zip in Drive WITHOUT a full recursive scan
    (globbing '/content/drive/MyDrive/**' walks the entire Drive over the
    network and can hang for many minutes). Try explicit paths first, then a
    depth- and count-bounded breadth-first search that prints progress."""
    if DRIVE_ZIP_PATH:
        if os.path.exists(DRIVE_ZIP_PATH):
            return DRIVE_ZIP_PATH
        raise FileNotFoundError(f"DRIVE_ZIP_PATH is set but not found: {DRIVE_ZIP_PATH}")

    target = "phase-2-python-sql.zip"
    # Fast, instant checks of the most likely spots (top of Drive + course folder).
    for cand in (
        f"/content/drive/MyDrive/{target}",
        f"/content/drive/MyDrive/Data Analysis and AI Automation Course Cohort 7/Dataset/{target}",
        f"/content/{target}",
    ):
        if os.path.exists(cand):
            return cand

    # Bounded BFS: depth <= 4, at most ~600 folders, skipping hidden dirs.
    print("Searching your Google Drive for phase-2-python-sql.zip ...")
    root, queue, scanned = "/content/drive/MyDrive", [("/content/drive/MyDrive", 0)], 0
    while queue:
        d, depth = queue.pop(0)
        hit = os.path.join(d, target)
        if os.path.exists(hit):
            return hit
        if depth >= 4:
            continue
        try:
            for e in os.scandir(d):
                if e.is_dir() and not e.name.startswith("."):
                    queue.append((e.path, depth + 1))
        except OSError:
            continue
        scanned += 1
        if scanned % 50 == 0:
            print(f"  ...scanned {scanned} folders")
        if scanned >= 600:
            break

    raise FileNotFoundError(
        "Could not quickly find phase-2-python-sql.zip in your Drive. Put the zip at the "
        "TOP of your Drive (My Drive) and re-run, or set DRIVE_ZIP_PATH at the top of this "
        "cell to its exact path.")


def _find_csv_dir():
    """Return the folder that actually contains olist_orders_dataset.csv."""
    roots = []
    env_dir = os.environ.get("OLIST_DATA_PATH", "")   # set by the pipeline validator
    if env_dir:
        roots.append(env_dir)
    if LOCAL_DATA_DIR:
        roots.append(LOCAL_DATA_DIR)

    if ON_COLAB:
        extract_path = "/content/olist_data"
        # unzip only the first time; reuse the extracted CSVs afterwards
        if not glob.glob(f"{extract_path}/**/olist_orders_dataset.csv", recursive=True):
            zip_path = _colab_find_zip()
            os.makedirs(extract_path, exist_ok=True)
            print(f"Unzipping {os.path.basename(zip_path)} ...")
            with zipfile.ZipFile(zip_path) as z:
                z.extractall(extract_path)
        roots.append(extract_path)
    else:
        # Local: search cwd (recursively) + a few common spots — never the whole
        # home dir (that recursive walk can be very slow). Set LOCAL_DATA_DIR if
        # your CSVs live elsewhere.
        roots += [os.getcwd(),
                  os.path.expanduser("~/Downloads"),
                  os.path.expanduser("~/Desktop"),
                  os.path.expanduser("~/olist")]

    for root in roots:
        if os.path.exists(os.path.join(root, "olist_orders_dataset.csv")):
            return root
        hits = glob.glob(os.path.join(root, "**", "olist_orders_dataset.csv"), recursive=True)
        if hits:
            return os.path.dirname(hits[0])

    raise FileNotFoundError(
        "Olist CSVs not found. Set LOCAL_DATA_DIR (local) or DRIVE_ZIP_PATH (Colab) at "
        "the top of this cell.")


DATA_DIR = _find_csv_dir()
print("Data folder:", DATA_DIR)

# Build a file-based SQLite DB shared by pandas (loading) and jupysql (querying).
DB_PATH = os.environ.get("OLIST_DB_PATH") or (
    "/content/olist.db" if ON_COLAB else os.path.join(tempfile.gettempdir(), "olist.db"))

tables = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "product_category_translation": "product_category_name_translation.csv",
}

conn = sqlite3.connect(DB_PATH)
for table_name, filename in tables.items():
    df = pd.read_csv(os.path.join(DATA_DIR, filename))
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    print(f"Loaded {table_name}: {len(df):,} rows")
conn.close()
print("\nDatabase ready.")

# On Colab, install jupysql so `%load_ext sql` loads it instead of the legacy
# ipython-sql (see header). Off Colab (local / pipeline validator) jupysql is
# already installed, so we skip the install and stay offline-safe.
if ON_COLAB:
    get_ipython().run_line_magic("pip", "install --quiet --upgrade jupysql")

get_ipython().run_line_magic("load_ext", "sql")

# Guard: if the legacy ipython-sql was already loaded earlier THIS session (e.g.
# an older cell ran first), the freshly installed jupysql cannot hot-swap in — a
# runtime restart is the only fix. jupysql exposes sql.connection.ConnectionManager;
# ipython-sql does not. Stop with a clear instruction instead of a later cryptic
# prettytable KeyError.
import sql.connection as _sqlconn
if not hasattr(_sqlconn, "ConnectionManager"):
    raise RuntimeError(
        "Legacy ipython-sql is active, not jupysql. On Colab: Runtime -> Restart session, "
        "then run THIS setup cell first (before any other cell). Locally: "
        "pip install --upgrade jupysql and restart the kernel."
    )

# Connect the %%sql magic to the SAME database file. autopandas=True is REQUIRED
# (see header). We connect with run_line_magic (not a literal `%sql` line) so the
# computed DB_PATH is interpolated correctly. Do NOT set SqlMagic.style.
get_ipython().run_line_magic("config", "SqlMagic.autopandas = True")
get_ipython().run_line_magic("config", "SqlMagic.feedback = 0")
get_ipython().run_line_magic("sql", f"sqlite:///{DB_PATH}")

# Verify (expected row counts — do not alter without re-running against data):
#   orders 99,441 | customers 99,441 | order_items 112,650 | order_payments 103,886
#   order_reviews 99,224 | products 32,951 | sellers 3,095 | product_category_translation 71


## Why this matters

Olist has 3,095 sellers on the platform, and Operations wants a leaderboard: who earns the
most, and by how much do the top few pull ahead of everyone else? A plain `GROUP BY` can
compute each seller's total revenue, but it cannot also *rank* every row against every other
row without collapsing the table down to one output row per seller — you'd lose the ability
to see, say, all ten order-item rows belonging to the top seller side by side with their rank.
That is exactly the gap a **window function** fills: it computes an aggregate or a rank *per
row*, using a "window" of other rows to compare against, but it never reduces the row count.
Today you'll use that idea three ways — to rank sellers, to build a running total through the
year, and to see precisely how ties are handled differently by two functions that look
almost identical.


## 1. `RANK() OVER (ORDER BY ...)` — ranking without collapsing rows

A window function has the shape `<function>() OVER (ORDER BY <expr> [PARTITION BY <expr>])`.
The `OVER` clause is what makes it a window function: it tells SQLite *"don't aggregate this
away — instead, look across a window of rows (here, all of them, ordered by revenue) and
compute a value for each row relative to that window."* `RANK()` assigns 1 to the row with
the highest value in the `ORDER BY`, 2 to the next, and so on — and here's the detail that
matters: if two rows tie for revenue, `RANK()` gives them the **same** rank number and then
**skips** the rank after it (1, 1, 3 — never 1, 1, 2). You still get one output row per
seller, exactly like `GROUP BY` would give you, but now that row also carries its rank.


In [ ]:
%%sql
-- Rank sellers by total revenue
SELECT seller_id,
       ROUND(SUM(price), 2) AS total_revenue,
       RANK() OVER (ORDER BY SUM(price) DESC) AS revenue_rank
FROM order_items
GROUP BY seller_id
LIMIT 10
-- Expected top row: seller_id = 4869f7a5dfa277a7dca6462dcf3b52b2, total_revenue = 229472.63, revenue_rank = 1


---
## 🤖 Using DeepSeek this week

The same prompt-then-verify protocol from Wednesday applies to window functions. They are
an especially easy place for an AI-drafted query to look right and be subtly wrong — a
missing `ORDER BY` inside `OVER()` silently turns every `RANK()` into `1`, and it is easy to
miss because the query still runs without error.

The protocol:
1. **Ask** DeepSeek: *"Write a SQLite query that ranks sellers in the `order_items` table by
   their total `price`, using `RANK() OVER`, showing the top 10."*
2. **Run** whatever it gives you.
3. **Verify** the number against something you already know is correct — Concept 1 above
   already told you the answer: the top seller is `4869f7a5dfa277a7dca6462dcf3b52b2` at
   R$229,472.63. If DeepSeek's version doesn't land on that seller and figure at rank 1,
   the query has a bug (most often: a missing or wrong `ORDER BY` inside `OVER()`).


In [ ]:
%%sql
-- Step 3 of the protocol: run the DeepSeek-drafted query, then check its top row
-- against the value you already verified in Concept 1 before trusting it.
SELECT seller_id,
       ROUND(SUM(price), 2) AS total_revenue,
       RANK() OVER (ORDER BY SUM(price) DESC) AS revenue_rank
FROM order_items
GROUP BY seller_id
ORDER BY revenue_rank
LIMIT 1
-- Expected: seller_id = 4869f7a5dfa277a7dca6462dcf3b52b2, total_revenue = 229472.63, revenue_rank = 1 (matches Concept 1)


## 2. `SUM(...) OVER (ORDER BY ...)` — a running total across time

Stack a `SUM()` inside `OVER (ORDER BY ...)` and you get a **running total**: each row's
value is the sum of itself and every row before it in the ordering. This is the query
Finance actually wants for a year-end report — not "how many orders in November" in
isolation, but "how many orders *so far* by the end of November." Notice the query still
needs `GROUP BY month` to collapse each day down to one row per month first; the window
function then runs a *second* pass over those monthly rows, accumulating as it goes. That's
the pattern: aggregate to the grain you want with `GROUP BY`, then let the window function
walk across the aggregated rows.


In [ ]:
%%sql
-- Running total of orders through 2017
SELECT strftime('%Y-%m', o.order_purchase_timestamp) AS month,
       COUNT(*) AS monthly_orders,
       SUM(COUNT(*)) OVER (ORDER BY strftime('%Y-%m', o.order_purchase_timestamp)) AS running_total
FROM orders o
WHERE strftime('%Y', o.order_purchase_timestamp) = '2017'
GROUP BY month
ORDER BY month
-- Expected rows: 2017-01 -> 800/800 | 2017-06 -> 3,245/14,611 | 2017-11 -> 7,544/39,428 | 2017-12 -> 5,673/45,101


## 3. `ROW_NUMBER()` vs `RANK()` — same syntax, different tie behavior

Both functions number rows according to an `ORDER BY`, and both are window functions with
identical syntax — the only difference is what happens on a tie. `ROW_NUMBER()` hands out a
strictly increasing sequence (1, 2, 3, 4, ...) no matter what — even two rows with the exact
same `order_count` get different numbers, decided arbitrarily by whichever row SQLite
processes first. `RANK()` instead gives tied rows the *same* number and then skips ahead
(1, 1, 3). Which one you want depends on the question: use `ROW_NUMBER()` when you need a
single row per "slot" (e.g. picking exactly one top row per group), and `RANK()` when ties
should visibly share a placing. Run both side by side and watch where — if anywhere among
these eight states — they disagree.


In [ ]:
%%sql
-- ROW_NUMBER: unique sequential number even with ties. RANK: ties share a rank.
-- No single verified numeric target here — compare the two columns directly and
-- watch for any row where row_num and rank_num diverge (that's a tie).
SELECT customer_state,
       COUNT(*) AS order_count,
       ROW_NUMBER() OVER (ORDER BY COUNT(*) DESC) AS row_num,
       RANK() OVER (ORDER BY COUNT(*) DESC) AS rank_num
FROM customers
GROUP BY customer_state
ORDER BY order_count DESC
LIMIT 8


## Going deeper — `PARTITION BY`: resetting the window per group

Every window function so far has ranked or summed across the **entire** result set — one
single window covering all rows. Add `PARTITION BY <column>` to the `OVER()` clause and the
window function instead resets at the start of every group, exactly like `GROUP BY` resets
an aggregate, except the underlying rows are never collapsed. Here, instead of ranking every
order item in the whole table by price, `PARTITION BY order_id` ranks the items **within
each order** — item 1 of order A is compared only against the other items of order A, never
against order B. This is the single most useful window-function extension you'll reach for:
"top N per group" is almost always a `PARTITION BY` question.


In [ ]:
%%sql
-- Rank each item within its own order by price (highest first) — PARTITION BY
-- resets the ranking at every new order_id, so item ranks never cross orders.
SELECT order_id,
       product_id,
       price,
       ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY price DESC) AS item_rank
FROM order_items
ORDER BY order_id, item_rank
LIMIT 10


## Common mistakes

**Mistake — filtering directly on a window function's result in `WHERE`.** `WHERE` runs
*before* window functions are computed, so SQLite has no `revenue_rank` value yet to compare
against at that stage — the query errors out (`revenue_rank` is not a recognized column at
the point `WHERE` runs). The fix is to compute the ranking first inside a `WITH` CTE, then
filter the *outer* query against the CTE's output, where the rank column already exists.


In [ ]:
%%sql
-- ── COMMON MISTAKE ──────────────────────────────────────────────────
-- WRONG — errors out, because WHERE cannot see a window function's own output:
--   SELECT seller_id, ROUND(SUM(price), 2) AS total_revenue,
--          RANK() OVER (ORDER BY SUM(price) DESC) AS revenue_rank
--   FROM order_items
--   GROUP BY seller_id
--   WHERE revenue_rank <= 3
-- CORRECT — compute the rank in a CTE, then filter the outer query against it:
WITH ranked AS (
    SELECT seller_id,
           ROUND(SUM(price), 2) AS total_revenue,
           RANK() OVER (ORDER BY SUM(price) DESC) AS revenue_rank
    FROM order_items
    GROUP BY seller_id
)
SELECT * FROM ranked WHERE revenue_rank <= 3
-- Expected top row: seller_id = 4869f7a5dfa277a7dca6462dcf3b52b2, total_revenue = 229472.63, revenue_rank = 1


## Mini-challenge — your turn

⏱ ~5–10 min

Concept 2 built a running total of monthly orders through 2017. Extend it: add one more
column, `running_pct`, expressing the running total as a percentage of the **full 2017
total** (45,101 orders) — `running_total * 100.0 / 45101`, rounded to 2 decimals. You only
need today's clauses: the same `SUM(...) OVER (ORDER BY ...)` from Concept 2, plus one more
arithmetic expression in the `SELECT` list.

**Expected:** the last row, December 2017, should show `running_pct = 100.0` — by definition,
the running total at the final month equals the full-year total.


In [ ]:
%%sql
-- ⏱ ~5-10 min — your turn! Replace the placeholder below with your own query.
SELECT 'write your query here' AS todo


## Session Summary

| Function | What it does | Example |
|---|---|---|
| `RANK() OVER (ORDER BY ...)` | ranks rows by a value; ties share a rank and skip the next number | `RANK() OVER (ORDER BY SUM(price) DESC)` |
| `SUM(...) OVER (ORDER BY ...)` | running total — each row accumulates every row before it | `SUM(COUNT(*)) OVER (ORDER BY month)` |
| `ROW_NUMBER() OVER (ORDER BY ...)` | strictly sequential numbering; ties get different numbers | `ROW_NUMBER() OVER (ORDER BY COUNT(*) DESC)` |
| `PARTITION BY` | resets the window per group, without collapsing rows | `OVER (PARTITION BY order_id ORDER BY price DESC)` |

Unlike `GROUP BY`, every window function above returned **all** the underlying rows — it
added a rank or a running total as an extra column instead of collapsing the table down to
one row per group. That's the whole idea of a window function: aggregate *and* keep the
detail.


---
### Closing the session — the group exercise

This week's **group exercise** brings subqueries and window functions together, combining
Wednesday's material with today's. In your groups you will:

1. **Find sellers above average revenue** — a scalar subquery in `WHERE`, exactly like
   Wednesday's Concept 1, but against seller totals instead of individual payments.
2. **Rank customer states by average review score** — join `orders`, `customers`, and
   `order_reviews`, then rank the result with `RANK()`, the same pattern as Concept 1 above.
3. **Build a running total of payment revenue through 2018** — the same `SUM(...) OVER
   (ORDER BY ...)` shape as Concept 2, applied to `order_payments` instead of `orders`, and
   to 2018 instead of 2017.
4. **Number each review within its review-score group with `ROW_NUMBER()`**, partitioned by
   `review_score` — the `PARTITION BY` idea from Going Deeper, applied to a different table.

Every one of today's four concepts reappears in one of these four tasks — this is the
session where subqueries and window functions stop being separate topics and become tools
you reach for together.
